# 06 Showcase data

This notebook gathers the results from notebooks 01 to 05 into `docs/data.js`, the only data file the website reads. Numbers on the site therefore come from the same tables as the Insight Report.

In [ ]:
%run 00_utils.ipynb

import hashlib, json
import pandas as pd

COMMUNITY_NAMES = {"C1": "HIV generics core", "C2": "TB generics", "C3": "Malaria, injectables and originators",
                   "C4": "TB patient kits", "C5": "Single-originator ARVs", "C6": "Arteether", "C7": "Pyronaridine-artesunate"}

## Product map

In [ ]:
# medicines and their suppliers in the main network spec
nodes = pd.read_csv(NETWORK / "product_nodes_analyzed_min2.csv", index_col=0)
main_links = pd.read_csv(NETWORK / "bipartite_edges_min2.csv")
main_proximity = pd.read_csv(NETWORK / "product_edges_min2.csv")

# firms are listed from largest to smallest spend so the map can name the main suppliers
suppliers = main_links.sort_values("spend_usd", ascending=False).groupby("product")["firm"].apply(list)

map_nodes = [{"id": name, "area": row.area, "community": row.community, "spend": round(row.spend_usd / 1e6, 1),
              "firms": suppliers.get(name, [])} for name, row in nodes.iterrows()]

# only strong ties are drawn so the map stays readable
strong_ties = main_proximity[main_proximity.phi >= 0.5]
map_edges = [[source, target] for source, target in zip(strong_ties.source, strong_ties.target)]

## Candidate suppliers

In [ ]:
# use the same links as the candidate model: every firm-product link that began by 2025
links_2025 = pd.read_csv(NETWORK / "bipartite_edges_min1.csv").query("first_year < 2026")
proximity = project_products(links_2025)
both_directions = pd.concat([proximity.rename(columns={"source": "product", "target": "neighbor"}),
                             proximity.rename(columns={"target": "product", "source": "neighbor"})])
firm_products = links_2025.groupby("firm")["product"].apply(set)


def closest_medicines(product, n_closest=5):
    """Return the medicines that share the most suppliers with this one."""
    rows = both_directions[both_directions["product"] == product].sort_values(["phi", "neighbor"], ascending=[False, True])
    return rows.head(n_closest)["neighbor"].tolist()


candidate_table = pd.read_csv(TABLES / "t12_candidate_suppliers.csv")

# closest outside firm for each product, read from the full priority table
priority_rows = pd.read_csv(TABLES / "t14_priority_medicines.csv").dropna(subset=["pqr_products"])
closest_density = {product: row.closest_density for row in priority_rows.itertuples() for product in row.pqr_products.split("; ")}

medicines = []
for product, group in candidate_table.groupby("product", sort=False):
    group = group.sort_values("rank")
    neighbors = closest_medicines(product)
    first = group.iloc[0]
    medicines.append({
        "name": product, "area": first.area, "spend": round(float(first.spend_2021_25_usd_m), 1),
        "current": first.current_suppliers.split("; "), "neighbors": neighbors,
        "reading": "nearby" if closest_density[product] >= 0.3 else "far", "closest": closest_density[product],
        "candidates": [{"firm": row.firm, "country": None if pd.isna(row.hq_country) else row.hq_country,
                        "p": round(float(row.p_entry_3y), 4), "density": round(float(row.density), 3),
                        "who": bool(row.who_pq_listed),
                        "supplies": [neighbor in firm_products.get(row.firm, set()) for neighbor in neighbors]}
                       for row in group.itertuples()]})

## Evidence and headline numbers

In [ ]:
network_summary = pd.read_csv(TABLES / "t8_network_summary.csv").set_index("metric")["value"]
null_draws = pd.read_csv(NETWORK / "null_model_min2.csv")
density_rates = pd.read_csv(TABLES / "t11b_entry_by_density.csv")
prediction = pd.read_csv(TABLES / "t11_entry_prediction.csv").set_index("model")
entry_panel = pd.read_csv(PROCESSED / "entry_panel.csv").query("firm_active_recent and prod_bought_recent")
test_rows = entry_panel[entry_panel.cutoff >= 2020]

priority = pd.read_csv(TABLES / "t14_priority_medicines.csv")
readings = priority.reading.value_counts()

stats = {
    "products": len(nodes), "communities": int(network_summary["communities"]), "modularity": float(network_summary["modularity_Q"]),
    "null_modularity": float(network_summary["null_Q_mean"]), "nmi": float(network_summary["NMI_area"]),
    "pairs": len(entry_panel), "entries": int(entry_panel.entry.sum()), "test_entries": int(test_rows.entry.sum()),
    "recall_controls": float(prediction.loc["Controls logit", "recall_top5%"]),
    "recall_density": float(prediction.loc["Density logit", "recall_top5%"]),
    "priority": len(priority), "nearby": int(readings["Nearby firm"]), "not_nearby": int(readings["No nearby firm"]),
    "no_purchases": int(readings["No recent purchases"]), "not_scored": int(readings["Not scored"]), "scored": len(medicines),
    "no_listing": int((priority.pq_firms == 0).sum()),
}

## Write the website data

## Investment game

In [ ]:
# the game follows the real TLD market: each round asks who to sign among companies that had not started yet
GAME_PRODUCT, ROUNDS, LAST_YEAR = "Dolutegravir + Lamivudine + Tenofovir DF - FDC", [2018, 2019, 2020], 2025
pqr = load_clean("pqr_medicines")
all_links = pqr[pqr.year.between(2005, LAST_YEAR)].groupby(["firm", "product"]).year.min().rename("first_year").reset_index()

# real deliveries: pills shipped to Global Fund buyers, shown as years of treatment at one pill a day
tld = pqr[(pqr["product"] == GAME_PRODUCT) & pqr.year.between(2017, LAST_YEAR)].copy()
tld["pills"] = tld.pack_qty * pd.to_numeric(tld["Nb of Suom in Pack"].astype(str).str.replace(",", ""), errors="coerce")
tld = tld[tld.pills > 0]
first_supply = tld.groupby("firm").year.min()
candidates_tld = first_supply[first_supply >= ROUNDS[0]].index

# WHO listings for TLD; listings through another regulator have no date
pq_tld = load_clean("pq_fpp")
pq_text = pq_tld["INN, Dosage Form and Strength"].fillna("").str.lower()
pq_tld = pq_tld[pq_text.str.contains("dolutegravir") & pq_text.str.contains("lamivudine") & pq_text.str.contains("tenofovir") & ~pq_text.str.contains("alafenamide")]
first_listing = pq_tld.groupby("firm").year.min()
listed_firms = set(pq_tld.firm)

# broader quality signals: every WHO-listed finished product, and WHO-listed active ingredients used in TLD
pq_all = load_clean("pq_fpp")
pq_ingredients = load_clean("pq_api")
pq_ingredients = pq_ingredients[pq_ingredients["INN"].fillna("").str.lower().str.contains("dolutegravir|lamivudine|tenofovir")]
product_areas = pqr.groupby("product")["area"].first()
firm_hq = pqr.groupby("firm")["hq_country"].first()


def as_of(firm, year):
    """Return what a reader could have known about a company at the start of a round."""
    state = all_links[all_links.first_year < year]
    proximity_edges = project_products(state)
    neighbors = pd.concat([proximity_edges.rename(columns={"source": "product", "target": "neighbor"}),
                           proximity_edges.rename(columns={"target": "product", "source": "neighbor"})])
    neighbors = neighbors[neighbors["product"] == GAME_PRODUCT]
    products = set(state[state.firm == firm]["product"])
    total = neighbors.phi.sum()
    if firm not in listed_firms:
        approval = "none"
    elif pd.isna(first_listing.get(firm)):
        approval = "undated"
    else:
        approval = "yes" if first_listing[firm] < year else "none"
    areas = {product_areas[product] for product in products}
    firm_listings = pq_all[pq_all.firm == firm]
    return {"approval": approval, "match": round(neighbors[neighbors.neighbor.isin(products)].phi.sum() / total, 3) if total else 0,
            "medicines": len(products), "hiv": sum(product_areas[product] == "HIV/ARV" for product in products), "areas": sorted(areas),
            "listings": int((firm_listings.year < year).sum()),
            "ingredients": int(pq_ingredients[(pq_ingredients.firm == firm) & (pq_ingredients.year < year)]["INN"].nunique())}


# fixed letters hide company names until the results; hashing keeps the order from following the alphabet
ordered = sorted(candidates_tld, key=lambda firm: hashlib.md5(firm.encode()).hexdigest())
years = list(range(ROUNDS[0], LAST_YEAR + 1))
country_ids = {country: i for i, country in enumerate(sorted(tld.country.unique()))}
game_firms = []
for letter, firm in zip("ABCDEFGHIJKLMNOP", ordered):
    rows = tld[tld.firm == firm]
    game_firms.append({
        "label": f"Company {letter}", "firm": firm, "country": None if pd.isna(firm_hq[firm]) else firm_hq[firm], "started": int(first_supply[firm]),
        "cost_year": round(float(rows.cost_usd.sum() / rows.pills.sum() * 365), 1),
        "by_year": [int(rows[rows.year == year].pills.sum() / 365) for year in years],
        "countries": [sorted({country_ids[c] for c in rows[rows.year == year].country}) for year in years],
        "known": {str(year): as_of(firm, year) for year in ROUNDS if first_supply[firm] >= year}})

market_rows = tld.assign(already=tld.groupby("firm").year.transform("min"))
market = {str(year): {"firms": int(market_rows[market_rows.already < year].firm.nunique()),
                      "last_year": int(tld[tld.year == year - 1].pills.sum() / 365)} for year in ROUNDS}

# the last step of the game uses three current WHO priority medicines
TODAY = [("Bedaquiline", "Bedaquiline (drug-resistant TB)"), ("Dihydroartemisinin+Piperaquine - FDC", "DHA-piperaquine (malaria)"),
         ("Artesunate+Pyronaridine tetraphosphate", "Pyronaridine-artesunate (malaria)")]
by_name = {medicine["name"]: medicine for medicine in medicines}
today = [{"plain": plain, "firms": sorted(({"firm": c["firm"], "density": c["density"], "who": c["who"]} for c in by_name[name]["candidates"]),
                                         key=lambda firm: -firm["density"])[:3]} for name, plain in TODAY]
game = {"rounds": ROUNDS, "years": years, "firms": game_firms, "market": market, "today": today, "n_countries": len(country_ids)}
print(pd.DataFrame([{k: f[k] for k in ["label", "firm", "started", "cost_year"]} | {"years": sum(f["by_year"]), "known": f["known"]["2018"] if "2018" in f["known"] else None} for f in game_firms]).to_string())
print(market)

In [ ]:
site_data = {"stats": stats, "communities": COMMUNITY_NAMES, "nodes": map_nodes, "edges": map_edges, "medicines": medicines, "game": game,
             "not_purchased": [{"name": row.molecules.replace("+", " + ").title(), "area": row.disease, "listed": int(row.pq_firms)} for row in priority[priority.reading == "No recent purchases"].itertuples()],
             "null": null_draws.Q.round(4).tolist(), "density": density_rates[["density_bin", "pairs", "entry_rate_%"]].values.tolist()}

docs = ROOT / "docs"
docs.mkdir(exist_ok=True)
(docs / "data.js").write_text("window.SITE_DATA = " + json.dumps(site_data, ensure_ascii=False) + ";\n")
print(stats)